# Chapter 1, Problem 20 — the Lorentz transformation in matrix form

**Problem.** Write the Lorentz transformation equations in matrix form.

**Starting point, Eq. (1.12)**, a boost with velocity $v$ along $x$:
$$
\bar t = \gamma t - \gamma v x,\qquad \bar x = -\gamma v t + \gamma x,\qquad \bar y = y,\qquad \bar z = z,\qquad \gamma = (1-v^2)^{-1/2}.
$$

## The matrix

Collect the coordinates into a column, in the order $(x^0, x^1, x^2, x^3) = (t, x, y, z)$. Each equation is one row: the coefficient of $t$ goes in column 0, of $x$ in column 1, and so on.
$$
\boxed{
\begin{pmatrix}\bar t\\ \bar x\\ \bar y\\ \bar z\end{pmatrix}
=
\begin{pmatrix}
\gamma & -\gamma v & 0 & 0\\
-\gamma v & \gamma & 0 & 0\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{pmatrix}
\begin{pmatrix} t\\ x\\ y\\ z\end{pmatrix}}
$$

Row by row: row 0 gives $\bar t = \gamma t - \gamma v x + 0y + 0z$ ✓; row 1 gives $\bar x = -\gamma v t + \gamma x$ ✓; rows 2–3 give $\bar y = y$, $\bar z = z$ ✓.

**Index form** (anticipating Chapter 2): with $\Lambda^{\bar\alpha}{}_{\beta}$ the entry in row $\bar\alpha$, column $\beta$,
$$
x^{\bar\alpha} = \sum_{\beta=0}^{3}\Lambda^{\bar\alpha}{}_{\beta}\,x^\beta,\qquad
\Lambda^{\bar0}{}_0 = \Lambda^{\bar1}{}_1 = \gamma,\ \ \Lambda^{\bar0}{}_1 = \Lambda^{\bar1}{}_0 = -\gamma v,\ \ \Lambda^{\bar2}{}_2 = \Lambda^{\bar3}{}_3 = 1,\ \text{others } 0.
$$
The same matrix transforms coordinate *differences* $\Delta x^\alpha$, since the map is linear. (This is Schutz eq. 2.3, which `schutz_gr.boost_matrix` implements.)

**Rapidity form** (Problem 19), $v = \tanh u$:
$$
\Lambda(u) = \begin{pmatrix}\cosh u & -\sinh u & 0 & 0\\ -\sinh u & \cosh u & 0 & 0\\ 0&0&1&0\\0&0&0&1\end{pmatrix}.
$$

## Properties that are easy to see in matrix form

1. **Inverse = boost with $-v$.** $\Lambda(v)\Lambda(-v) = I$. Top-left block: $\gamma^2 - \gamma^2v^2 = 1$ on the diagonal, $\gamma^2 v - \gamma^2 v = 0$ off it. Physically, $\mathcal O$ moves at $-v$ relative to $\bar{\mathcal O}$.
2. **Symmetric:** $\Lambda^T = \Lambda$ (true for a pure boost along an axis, not for general Lorentz transformations).
3. **Determinant 1:** $\gamma^2 - \gamma^2 v^2 = 1$ — orientation and 4-volume preserved.
4. **Invariance of the interval.** With $\eta = \mathrm{diag}(-1,1,1,1)$, $\Delta s^2 = \Delta x^T\eta\,\Delta x$, and $\Delta\bar s^2 = \Delta x^T(\Lambda^T\eta\Lambda)\Delta x$. Interval invariance is the matrix statement
$$
\Lambda^T\eta\,\Lambda = \eta ,
$$
which *defines* the Lorentz group (compare rotations: $R^TR = I$). In Problem 7's language, $M = \Lambda^T\eta\Lambda$, and here $M = \eta$.
5. **Composition** is matrix multiplication: $\Lambda(v_2)\Lambda(v_1) = \Lambda\!\left(\frac{v_1+v_2}{1+v_1v_2}\right)$, which is Eq. (1.13) — or in rapidities, $\Lambda(u_2)\Lambda(u_1) = \Lambda(u_1 + u_2)$ (Problem 18).

In [1]:
import sympy as sp
from schutz_gr import boost_matrix
v, v1, v2, u, u1, u2 = sp.symbols("v v1 v2 u u1 u2", real=True)
L = boost_matrix(v)
display(L)
eta = sp.diag(-1, 1, 1, 1)
print("Λ(v)Λ(−v) = I:", sp.simplify(L * boost_matrix(-v)) == sp.eye(4))
print("Λ symmetric:", L == L.T, "   det Λ =", sp.simplify(L.det()))
print("Λᵀ η Λ = η:", sp.simplify(L.T * eta * L - eta) == sp.zeros(4))

comp = sp.simplify(boost_matrix(v2) * boost_matrix(v1))
w = (v1 + v2) / (1 + v1 * v2)
print("Λ(v2)Λ(v1) = Λ((v1+v2)/(1+v1 v2)):",
      sp.simplify((comp - boost_matrix(w)).subs({v1: sp.Rational(3, 10), v2: sp.Rational(-7, 13)})) == sp.zeros(4))

Lu = lambda r: sp.Matrix([[sp.cosh(r), -sp.sinh(r), 0, 0], [-sp.sinh(r), sp.cosh(r), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
print("Λ(u2)Λ(u1) = Λ(u1+u2):", (Lu(u2)*Lu(u1) - Lu(u1 + u2)).applyfunc(lambda e: sp.simplify(e.rewrite(sp.exp))) == sp.zeros(4))

Matrix([
[ 1/sqrt(1 - v**2), -v/sqrt(1 - v**2), 0, 0],
[-v/sqrt(1 - v**2),  1/sqrt(1 - v**2), 0, 0],
[                0,                 0, 1, 0],
[                0,                 0, 0, 1]])

Λ(v)Λ(−v) = I: True
Λ symmetric: True    det Λ = 1
Λᵀ η Λ = η: True
Λ(v2)Λ(v1) = Λ((v1+v2)/(1+v1 v2)): True


Λ(u2)Λ(u1) = Λ(u1+u2): True


In [2]:
# numeric example: an event and its transform, and the interval before/after
import numpy as np
from schutz_gr import interval
Ln = boost_matrix(0.6)
X = np.array([2.0, 1.0, -0.5, 3.0])
Xb = Ln @ X
print("Λ(0.6) =\n", np.round(Ln, 4))
print("x =", X, " → x̄ =", np.round(Xb, 4))
print("Δs² before =", interval(X), "  after =", round(interval(Xb), 12))

Λ(0.6) =
 [[ 1.25 -0.75  0.    0.  ]
 [-0.75  1.25  0.    0.  ]
 [ 0.    0.    1.    0.  ]
 [ 0.    0.    0.    1.  ]]
x = [ 2.   1.  -0.5  3. ]  → x̄ = [ 1.75 -0.25 -0.5   3.  ]
Δs² before = 6.25   after = 6.25
